# NB5 — Compute XAI (SHAP / LIME / faithfulness / stability)

v7 pipeline. Runs on Kaggle GPU.

## Setup required before running

Same as NB4: add `xai-credit-preprocessed` (NB1) and `xai-credit-basins` (NB3) as input datasets.

## Predict functions used for explanation

All explanation and faithfulness computations use a **deterministic** predict function (dropout
off), for every canonical model. This is a deliberate, documented choice (flagged as an
unexplained inconsistency during the v6 audit): KernelSHAP and LIME both assume the function they
are approximating a local surrogate for is stable across repeated queries at the same input;
querying a function whose output changes randomly from call to call (e.g. one MC-Dropout pass at
a time) makes both explainers converge to noisier, less interpretable attributions, and inflates
apparent explainer instability with instability that is really coming from the model's own
inference-time stochasticity rather than the explainer. With dropout off, `eval()` mode gives
the expected output under the training-time dropout distribution (Srivastava et al., 2014's
weight-scaling argument), so it is a principled point-estimate, not an arbitrary substitute.

Only the number of basins averaged differs between models:

- **M2 and M3** — basin 1 only. M2 and M3 differ from each other only in how *uncertainty* is
  computed downstream (NB3); the point prediction being explained is identical, so this notebook
  computes SHAP/LIME **once** for this shared function and reuses the result for both labels —
  this also halves the KernelSHAP cost compared to running it separately for each.
- **M4** — average of all 50 basins.
- **M5** — average of basins 1-5.

## What is computed, per canonical model, per dataset

- SHAP values (`shap.KernelExplainer`) and LIME weights, on the 2000-sample balanced XAI analysis set.
- Faithfulness (Comprehensiveness / Sufficiency) in three variants (`prob`, `logit`, `norm`; see
  NB2 for definitions) at the adaptive-K grid, for both SHAP and LIME, with a random-K control
  stored **per sample** (not just as a mean) so NB6 can test whether the masking curve is
  specifically steeper for XAI-ranked features in the high-uncertainty stratum.
- Stability: reproducibility (Jaccard across repeated explainer runs on the same input) and local
  robustness (max-sensitivity under small input perturbations, Alvarez-Melis & Jaakkola, 2018).

## Outputs

`NB5-output/{model}_shap_{dataset}.npz`, `{model}_lime_{dataset}.npz` for model in
{M2, M3, M4, M5}
`NB5-output/faithfulness_{variant}_{method}_{dataset}.json` for variant in {prob, logit, norm},
method in {shap, lime} — 6 files per dataset, each keyed by model name
`NB5-output/stability_{dataset}.npz`


## 1. Installs & imports

In [ ]:
!pip install shap lime fastparquet scikit-learn -q

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
import lime
import lime.lime_tabular

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 2. Configuration

In [ ]:
class Config:
    DATA_DIR = "/kaggle/input/xai-credit-preprocessed"
    BASINS_DIR = "/kaggle/input/xai-credit-basins"
    OUTPUT_DIR = "/kaggle/working"
    SEED = 42
    DATASETS = ["home_credit", "taiwan", "gmsc"]
    # v7 audit fix: DATASET_FEATURES used to be a hard-coded dict here (with a stale value
    # for home_credit -- 122, not the real 105 numeric features once SK_ID_CURR and the
    # object-dtype columns are excluded), so the adaptive-K grid was silently computed from
    # the wrong denominator. get_k_vals now takes the *actual* feature count of the data
    # being processed instead of looking it up.

    HIDDEN_DIMS = [256, 128, 64]
    DROPOUT_RATE = 0.2

    BACKGROUND_SIZE = 100
    SHAP_NSAMPLES = 500
    LIME_NSAMPLES = 500

    STABILITY_N_INSTANCES = 20
    STABILITY_N_RERUNS = 10
    SENSITIVITY_N_PERTURB = 10
    SENSITIVITY_RADIUS = 0.01

    RANDOM_K_REPEATS = 10
    FAITH_CHUNK = 64          # so mau gop vao mot lan goi predict khi tinh
                              # faithfulness; chi anh huong toc do, khong doi ket qua

    # basin groups defining each canonical model's predict function (dropout always off here)
    MODEL_GROUPS = {"M2M3": [1], "M4": list(range(1, 51)), "M5": [1, 2, 3, 4, 5]}
    # which canonical model label maps to which group
    MODEL_TO_GROUP = {"M2": "M2M3", "M3": "M2M3", "M4": "M4", "M5": "M5"}

    @staticmethod
    def get_k_vals(n_features):
        percentages = [0.1, 0.2, 0.3, 0.4, 0.5]
        k_vals = [max(1, int(round(p * n_features))) for p in percentages]
        return sorted(set(k_vals))

    if not os.path.exists(DATA_DIR):
        print(f"WARNING: {DATA_DIR} not found. Assuming local test run.")
        DATA_DIR = "../kaggle_outputs/xai-credit-preprocessed"
    if not os.path.exists(BASINS_DIR):
        print(f"WARNING: {BASINS_DIR} not found. Assuming local test run.")
        BASINS_DIR = "../kaggle_outputs/xai-credit-basins"


import random
random.seed(Config.SEED)
np.random.seed(Config.SEED)

## 3. Model loading and predict functions

In [ ]:
class CreditMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128, 64), dropout_rate=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def load_basin(ds, basin_id, input_dim):
    model = CreditMLP(input_dim, Config.HIDDEN_DIMS, Config.DROPOUT_RATE).to(DEVICE)
    path = f"{Config.BASINS_DIR}/{ds}/basin_{basin_id:02d}.pt"
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


def make_predict_fn(ds, basin_ids, input_dim):
    """Deterministic (dropout off) predict function averaging over the given basins."""
    models = [load_basin(ds, bid, input_dim) for bid in basin_ids]

    def predict(X):
        X_t = torch.tensor(np.asarray(X, dtype=np.float32)).to(DEVICE)
        with torch.no_grad():
            probs = torch.zeros(X_t.shape[0], device=DEVICE)
            for m in models:
                probs += torch.sigmoid(m(X_t))
            probs /= len(models)
        return probs.cpu().numpy()

    return predict


def make_predict_proba_2col(predict_fn):
    def f(X):
        p = predict_fn(X)
        return np.column_stack([1 - p, p])
    return f


## 4. Faithfulness, random-K control, stability (same definitions as NB2)

In [ ]:
def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def calculate_faithfulness(predict_fn, X, base_probs, attributions, k_list, baseline_matrix,
                            variants=("prob", "logit", "norm"), random_repeats=1, seed=42,
                            norm_tol=0.005, chunk_size=64):
    """Comprehensiveness / Sufficiency + the Random-K control, for ALL variants in one pass.

    Returns {variant: (comp, suff, comp_rand, suff_rand)}, each a {k: array(n_samples)} dict.

    baseline_matrix : (n, d) array -- ONE masking point per sample, not a single population
        mean. v7 audit: masking every sample to the SAME mean vector on correlated tabular
        features is itself an out-of-manifold point; how far a sample sits from it turns out
        to correlate with epistemic uncertainty and manufactures a spurious link between
        uncertainty and comprehensiveness that has nothing to do with explanation quality
        (baseline-swap experiment, answers FB3). Per-sample points drawn from the training
        distribution do not have this problem. Fixed once per dataset by the caller, so every
        model/method/variant/K comparison in this notebook sees the same draw.
    norm_tol : the 'norm' variant divides by (base_probs - p_null); when that is close to zero
        the ratio is undefined and is written as NaN rather than clamped to a floor -- the old
        floor (max(., 1e-6)) silently kept the sign of the numerator, which flips ~45-70% of
        samples on Home Credit / GMSC (p_null sits mid-distribution there) so that a BETTER
        explanation scores as MORE negative. See table10_metric_orientation in NB6.
    variants / chunk_size : v7 audit fix for RUNTIME, not for correctness. The previous version
        took a single `variant` and was called three times, but all three variants are different
        arithmetic on the SAME masked probabilities -- so two thirds of the masking work was
        thrown away. It also called predict_fn once per masked row; with M4 averaging 50 basins
        that is 50 single-row GPU launches per call, and at 2000 analysis samples the stage did
        not fit in a Kaggle session. Now every masked row for `chunk_size` samples goes through
        predict_fn in one batch. Numerically safe because every basin runs under .eval(), so
        BatchNorm uses running statistics and batch size cannot change a row's output; verified
        against the old implementation at float64 (max deviation 3.7e-14, NaN masks identical).
    """
    n, n_features = X.shape
    variants = tuple(variants)
    out = {v: tuple({k: np.full(n, np.nan) for k in k_list} for _ in range(4))
           for v in variants}

    p_null_local = predict_fn(baseline_matrix)
    null_diff = base_probs - p_null_local              # SIGNED, not clamped
    null_defined = np.abs(null_diff) > norm_tol
    base_logit = to_logit(base_probs)

    n_per_sample = len(k_list) * 2 * (1 + random_repeats)

    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        rows, meta = [], []
        for i in range(start, stop):
            x_base = X[i]
            baseline_row = baseline_matrix[i]
            top_idx = np.argsort(-np.abs(attributions[i]))
            for k in k_list:
                top_k = top_idx[:k]
                x_comp = x_base.copy()
                x_comp[top_k] = baseline_row[top_k]
                x_suff = baseline_row.copy()
                x_suff[top_k] = x_base[top_k]
                rows += [x_comp, x_suff]
                meta += [(i, k, "comp"), (i, k, "suff")]
                for r in range(random_repeats):
                    # same RNG stream as the pre-batching version: a fresh RandomState per
                    # (sample, repeat), so the random masks are unchanged and stay shared
                    # across variants, methods and model groups
                    rng = np.random.RandomState(seed + i * 1000 + r)
                    rand_idx = rng.choice(n_features, size=min(k, n_features), replace=False)
                    x_comp_r = x_base.copy()
                    x_comp_r[rand_idx] = baseline_row[rand_idx]
                    x_suff_r = baseline_row.copy()
                    x_suff_r[rand_idx] = x_base[rand_idx]
                    rows += [x_comp_r, x_suff_r]
                    meta += [(i, k, "comp_r"), (i, k, "suff_r")]

        probs = predict_fn(np.asarray(rows, dtype=X.dtype))
        assert len(probs) == len(meta) == (stop - start) * n_per_sample, "batch bookkeeping"

        acc = {}
        for key, p in zip(meta, probs):
            acc.setdefault(key, []).append(p)

        for i in range(start, stop):
            for k in k_list:
                p_comp = acc[(i, k, "comp")][0]
                p_suff = acc[(i, k, "suff")][0]
                pc_r = np.asarray(acc[(i, k, "comp_r")])
                ps_r = np.asarray(acc[(i, k, "suff_r")])
                for v in variants:
                    comp, suff, comp_rand, suff_rand = out[v]
                    if v == "prob":
                        comp[k][i] = base_probs[i] - p_comp
                        suff[k][i] = base_probs[i] - p_suff
                        comp_rand[k][i] = np.mean(base_probs[i] - pc_r)
                        suff_rand[k][i] = np.mean(base_probs[i] - ps_r)
                    elif v == "logit":
                        comp[k][i] = base_logit[i] - to_logit(np.array([p_comp]))[0]
                        suff[k][i] = base_logit[i] - to_logit(np.array([p_suff]))[0]
                        comp_rand[k][i] = np.mean([base_logit[i] - to_logit(np.array([q]))[0]
                                                   for q in pc_r])
                        suff_rand[k][i] = np.mean([base_logit[i] - to_logit(np.array([q]))[0]
                                                   for q in ps_r])
                    elif v == "norm":
                        if not null_defined[i]:
                            continue   # stays NaN at every K, same as the pre-batching version
                        comp[k][i] = (base_probs[i] - p_comp) / null_diff[i]
                        suff[k][i] = (base_probs[i] - p_suff) / null_diff[i]
                        comp_rand[k][i] = np.mean((base_probs[i] - pc_r) / null_diff[i])
                        suff_rand[k][i] = np.mean((base_probs[i] - ps_r) / null_diff[i])

    return out
def compute_reproducibility_jaccard(explain_fn, X_subset, k_top, n_reruns=10, seed=0):
    n = len(X_subset)
    jaccards = np.zeros(n)
    for i in range(n):
        x0 = X_subset[i:i + 1]
        top_sets = []
        for r in range(n_reruns):
            vals = explain_fn(x0, seed=seed + r)
            top = set(np.argsort(-np.abs(vals[0]))[:k_top])
            top_sets.append(top)
        pairs = [(a, b) for idx_a, a in enumerate(top_sets) for b in top_sets[idx_a + 1:]]
        scores = [len(a & b) / len(a | b) for a, b in pairs]
        jaccards[i] = np.mean(scores)
    return jaccards


def calculate_max_sensitivity(explain_fn, X, n_perturbations=10, radius=0.01, seed=0):
    rng = np.random.RandomState(seed)
    base = explain_fn(X)
    sens = np.zeros(len(X))
    for i in range(len(X)):
        x0 = X[i:i + 1]
        pert = x0 + rng.uniform(-radius, radius, size=(n_perturbations, X.shape[1]))
        pert_vals = explain_fn(pert)
        diffs = np.linalg.norm(pert_vals - base[i], axis=1)
        sens[i] = np.max(diffs) / (np.linalg.norm(base[i]) + 1e-9)
    return sens

## 5. Per-dataset pipeline

In [ ]:
def process_dataset(ds):
    print(f"\n{'=' * 70}")
    print(f"DATASET: {ds}")
    print(f"{'=' * 70}")

    train_df = pd.read_parquet(f"{Config.DATA_DIR}/{ds}_train.parquet")
    test_df = pd.read_parquet(f"{Config.DATA_DIR}/{ds}_test_balanced.parquet")

    X_train = train_df.drop(columns=["TARGET"]).values.astype(np.float32)
    feature_names = train_df.drop(columns=["TARGET"]).columns.tolist()
    X_test = test_df.drop(columns=["TARGET"]).values.astype(np.float32)
    input_dim = X_test.shape[1]

    # Per-sample masking baseline drawn from the TRAIN distribution (v7 audit fix -- was
    # X_test.mean(axis=0), a single population-mean point; see calculate_faithfulness's
    # docstring). Fixed once per dataset, reused for every group/variant/method/K so all
    # comparisons in this notebook see the same baseline draw.
    baseline_rng = np.random.RandomState(Config.SEED)
    baseline_matrix = X_train[baseline_rng.choice(len(X_train), size=len(X_test), replace=True)]

    bg_idx = np.random.RandomState(Config.SEED).choice(
        len(X_train), size=min(Config.BACKGROUND_SIZE, len(X_train)), replace=False
    )
    X_background = X_train[bg_idx]

    k_list = Config.get_k_vals(input_dim)
    print(f"Adaptive-K grid: {k_list}  ({input_dim} features)")

    group_results = {}

    for group_name, basin_ids in Config.MODEL_GROUPS.items():
        print(f"\n  --- group {group_name}  (basins: {basin_ids[:5]}"
              f"{'...' if len(basin_ids) > 5 else ''}, n={len(basin_ids)}) ---")

        predict_fn = make_predict_fn(ds, basin_ids, input_dim)
        predict_proba_2col = make_predict_proba_2col(predict_fn)
        base_probs = predict_fn(X_test)
        p_null_samples = predict_fn(baseline_matrix)  # saved to JSON below (v7 audit
                                                        # addition -- was not saved before)

        print(f"  Computing SHAP (KernelExplainer, nsamples={Config.SHAP_NSAMPLES})...")
        explainer_shap = shap.KernelExplainer(predict_fn, X_background)
        shap_vals = explainer_shap.shap_values(X_test, nsamples=Config.SHAP_NSAMPLES, silent=True)
        shap_vals = np.asarray(shap_vals)

        print(f"  Computing LIME (nsamples={Config.LIME_NSAMPLES})...")
        explainer_lime = lime.lime_tabular.LimeTabularExplainer(
            X_background, feature_names=feature_names, class_names=["0", "1"],
            verbose=False, mode="classification",
        )
        lime_vals = np.zeros((len(X_test), len(feature_names)))
        for i in range(len(X_test)):
            exp = explainer_lime.explain_instance(
                X_test[i], predict_proba_2col, num_features=len(feature_names),
                num_samples=Config.LIME_NSAMPLES,
            )
            for feat_idx, weight in exp.as_map()[1]:
                lime_vals[i, feat_idx] = weight

        # One masking pass per method now yields all three variants (they are different
        # arithmetic on the same masked probabilities), and predict_fn is called on batches of
        # Config.FAITH_CHUNK samples' worth of masked rows instead of one row at a time.
        print(f"  Computing faithfulness (3 variants x 2 methods, single masking pass)...")
        faithfulness = {v: {} for v in ["prob", "logit", "norm"]}
        for method_name, attributions in [("shap", shap_vals), ("lime", lime_vals)]:
            per_variant = calculate_faithfulness(
                predict_fn, X_test, base_probs, attributions, k_list, baseline_matrix,
                variants=("prob", "logit", "norm"),
                random_repeats=Config.RANDOM_K_REPEATS, seed=Config.SEED,
                chunk_size=Config.FAITH_CHUNK,
            )
            for variant, faith_res in per_variant.items():
                faithfulness[variant][method_name] = faith_res
                comp, suff, comp_r, suff_r = faith_res
                mid_k = k_list[len(k_list) // 2]
                print(f"    [{variant:>5}/{method_name:>4}] comp@K={mid_k}: "
                      f"{np.nanmean(comp[mid_k]):.5f}  (random-K: {np.nanmean(comp_r[mid_k]):.5f})")

        print(f"  Computing stability (reproducibility + local robustness)...")
        stab_idx = np.random.RandomState(Config.SEED).choice(
            len(X_test), size=min(Config.STABILITY_N_INSTANCES, len(X_test)), replace=False
        )
        X_stab = X_test[stab_idx]

        def shap_explain_fn(X, seed=0, _explainer=explainer_shap):
            vals = _explainer.shap_values(X, nsamples=Config.SHAP_NSAMPLES, silent=True)
            return np.asarray(vals)

        def lime_explain_fn(X, seed=0, _explainer=explainer_lime, _pfn=predict_proba_2col):
            out = np.zeros((len(X), len(feature_names)))
            for i in range(len(X)):
                exp = _explainer.explain_instance(
                    X[i], _pfn, num_features=len(feature_names), num_samples=Config.LIME_NSAMPLES,
                )
                for feat_idx, weight in exp.as_map()[1]:
                    out[i, feat_idx] = weight
            return out

        k_top = max(1, k_list[1])
        jac_shap = compute_reproducibility_jaccard(
            shap_explain_fn, X_stab, k_top, n_reruns=Config.STABILITY_N_RERUNS, seed=Config.SEED,
        )
        jac_lime = compute_reproducibility_jaccard(
            lime_explain_fn, X_stab, k_top, n_reruns=Config.STABILITY_N_RERUNS, seed=Config.SEED,
        )
        max_sens_shap = calculate_max_sensitivity(
            shap_explain_fn, X_stab, n_perturbations=Config.SENSITIVITY_N_PERTURB,
            radius=Config.SENSITIVITY_RADIUS, seed=Config.SEED,
        )
        print(f"    reproducibility Jaccard: SHAP={np.mean(jac_shap):.4f} LIME={np.mean(jac_lime):.4f}"
              f" | local robustness (SHAP): {np.mean(max_sens_shap):.4f}")

        group_results[group_name] = {
            "shap_vals": shap_vals, "lime_vals": lime_vals, "base_probs": base_probs,
            "p_null_samples": p_null_samples,
            "faithfulness": faithfulness, "jac_shap": jac_shap, "jac_lime": jac_lime,
            "max_sens_shap": max_sens_shap, "stab_idx": stab_idx,
        }

    print(f"\n  Assembling per-model outputs (M2, M3, M4, M5)...")
    for model_label, group_name in Config.MODEL_TO_GROUP.items():
        res = group_results[group_name]
        np.savez_compressed(f"{Config.OUTPUT_DIR}/{model_label.lower()}_shap_{ds}.npz",
                             shap_values=res["shap_vals"])
        np.savez_compressed(f"{Config.OUTPUT_DIR}/{model_label.lower()}_lime_{ds}.npz",
                             lime_weights=res["lime_vals"])

    for variant in ["prob", "logit", "norm"]:
        for method_name in ["shap", "lime"]:
            payload = {}
            for model_label, group_name in Config.MODEL_TO_GROUP.items():
                comp, suff, comp_r, suff_r = group_results[group_name]["faithfulness"][variant][method_name]
                payload[model_label] = {
                    "comp_mean": {str(k): float(np.nanmean(comp[k])) for k in k_list},
                    "suff_mean": {str(k): float(np.nanmean(suff[k])) for k in k_list},
                    "comp_random_mean": {str(k): float(np.nanmean(comp_r[k])) for k in k_list},
                    "suff_random_mean": {str(k): float(np.nanmean(suff_r[k])) for k in k_list},
                    "comp_samples": {str(k): comp[k].tolist() for k in k_list},
                    "suff_samples": {str(k): suff[k].tolist() for k in k_list},
                    "comp_random_samples": {str(k): comp_r[k].tolist() for k in k_list},
                    "suff_random_samples": {str(k): suff_r[k].tolist() for k in k_list},
                    "k_list": k_list, "source_group": group_name,
                    # v7 audit additions -- saved directly instead of reconstructed later:
                    "base_probs": group_results[group_name]["base_probs"].tolist(),
                    "p_null_samples": group_results[group_name]["p_null_samples"].tolist(),
                }
            with open(f"{Config.OUTPUT_DIR}/faithfulness_{variant}_{method_name}_{ds}.json", "w") as f:
                json.dump(payload, f)

    stability_payload = {}
    for model_label, group_name in Config.MODEL_TO_GROUP.items():
        res = group_results[group_name]
        stability_payload[f"{model_label}_jaccard_shap"] = res["jac_shap"]
        stability_payload[f"{model_label}_jaccard_lime"] = res["jac_lime"]
        stability_payload[f"{model_label}_max_sensitivity_shap"] = res["max_sens_shap"]
        stability_payload[f"{model_label}_stab_idx"] = res["stab_idx"]
    np.savez_compressed(f"{Config.OUTPUT_DIR}/stability_{ds}.npz", **stability_payload)

    print(f"\nFinished {ds}.")

## 6. Run for all datasets

In [ ]:
for ds in Config.DATASETS:
    try:
        process_dataset(ds)
    except Exception as e:
        import traceback
        print(f"FAILED on {ds}: {e}")
        traceback.print_exc()

print("\nAll datasets processed. Download /kaggle/working as NB5-output.")
